In [2]:
import sys
sys.path.append("../")

from src.finman.utils.pdf import extract_text_from_pdf
from src.finman.expenses import tbank, ozon, sber, yapay
from src.finman.utils.gsheet import GSheetWorker

sheet_id = "1GqwNPuOtQUXfltK6F4oHpx-U1BqBgiDZfC5YfDcJyto"
gcreds_file = "../secrets/finman-433017-da308d823497.json"

gsw = GSheetWorker(gcreds_file)

In [149]:
trans_col = "Сумма операции в валюте карты"
date_col = "Дата операции"
time_col = "Время операции"
spreadsheet_name = "t-bank (main)"
parse_operations = tbank.parse_operations
extract_totals = tbank.extract_total_operations

gs_df = gsw.get_df(sheet_id, spreadsheet_name)

# ChatGPTшный метод

In [150]:
transactions_df = gs_df.copy()

In [151]:
import re
# Функция для фильтрации текста
def clean_description(description):
    # Убираем "Оплата в"
    description = re.sub(r'\bОплата в\b', '', description, flags=re.IGNORECASE)
    # Убираем паттерны "город + имя города + rus"
    description = re.sub(r'\b(?:gorod|city)?\s*(Moscow|Moskva|Sankt-Peter|[А-Яа-я]+)\s*(rus|ru)\b', '', description, flags=re.IGNORECASE)
    # Убираем лишние пробелы
    description = re.sub(r'\s+', ' ', description).strip()
    return description


In [152]:
# Apply cleaning to the description column
transactions_df['Описание операции'] = transactions_df['Описание операции'].apply(clean_description)

In [153]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


# Фильтруем транзакции без категории
missing_category = transactions_df[transactions_df['Категория'].isnull()]

# Транзакции с уже установленной категорией
labeled_data = transactions_df[transactions_df['Категория'].notnull()]

# Применим TF-IDF для текстового анализа
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(transactions_df['Описание операции'])

# Создадим копию данных для заполнения категорий
transactions_filled = transactions_df.copy()

In [154]:
missing_category.head(20)

,id,Дата операции,Время операции,Сумма операции в валюте карты,Валюта карты,Сумма в валюте операции,Валюта операции,Описание операции,Номер карты,Категория,Подкатегория,Долг,Хэштэг,Комментарий
129,b87edc4123,"08.10, Tue",2024-10-08 18:00,-1700,₽,-1700,₽,PE SPIRKOV IVAN SERGEEVICH g,5037,None,None,None,None,None
329,5d884e89a3,"02.09, Mon",2024-09-02 18:38,1000,₽,1000,₽,Внутрибанковский перевод с договора,5259,None,None,None,None,None
340,ba616b4c7d,"31.08, Sat",2024-08-31 21:24,-700,₽,-700,₽,Внешний перевод по номеру телефона +79645084158,5037,None,None,None,None,None
343,d6caff64dd,"31.08, Sat",2024-08-31 21:11,-283.5,₽,-283.5,₽,YM*weasel g,5037,None,None,None,None,None
344,72430a4884,"31.08, Sat",2024-08-31 21:04,-1500,₽,-1500,₽,Внешний перевод по номеру телефона +79645084158,5037,None,None,None,None,None
346,d02a8b49fe,"31.08, Sat",2024-08-31 17:42,-299,₽,-299,₽,Оплата услуг Tinkoff.tinkoff- bundle,5037,None,None,None,None,None
347,df8c3ed861,"31.08, Sat",2024-08-31 13:45,-400,₽,-400,₽,ONEBREW_KITCHEN,1665,None,None,None,None,None
348,2abb62db9d,"31.08, Sat",2024-08-31 13:03,-380,₽,-380,₽,ONEBREW_KITCHEN,1665,None,None,None,None,None
349,bbf7cd2918,"31.08, Sat",2024-08-31 12:45,-1220,₽,-1220,₽,ONEBREW_KITCHEN,1665,None,None,None,None,None
350,eb828c3462,"31.08, Sat",2024-08-31 10:17,-300,₽,-300,₽,OCHAKOVO,1665,None,None,None,None,None


In [155]:
# index = 350
index = 343

In [156]:
missing_category.at[index, 'Описание операции'] 

'YM*weasel g'

In [157]:
similarity = cosine_similarity(tfidf_matrix[index], tfidf_matrix).flatten()

# similarity = similarity[similarity > 0.5]
    
# Отсортируем по схожести
# similar_indices = similarity.argsort()[::-1][0:6]  # 5 наиболее похожих
# similar_transactions = transactions_df.iloc[similar_indices]
# Фильтруем только те, у которых есть категория
# similar_labeled = similar_transactions[similar_transactions['Категория'].notnull()]

similar_indices = similarity.argsort()[::-1]
similar_transactions = transactions_df.iloc[similar_indices]
similar_labeled = similar_transactions.loc[similar_transactions['Категория'].notnull() & (similarity[similar_indices] > 0.5)].head(3)

if not similar_labeled.empty and similar_labeled['Категория'].nunique() == 1 and similar_labeled['Подкатегория'].nunique() == 1:
    # Находим наиболее частую категорию и подкатегорию
    category = similar_labeled['Категория'].mode().iloc[0]
    subcategory = similar_labeled['Подкатегория'].mode().iloc[0]
    # return category, subcategory

In [158]:
similarity[147]

0.41460294827317856

In [159]:
similar_labeled

,id,Дата операции,Время операции,Сумма операции в валюте карты,Валюта карты,Сумма в валюте операции,Валюта операции,Описание операции,Номер карты,Категория,Подкатегория,Долг,Хэштэг,Комментарий
158,e696788b65,"01.10, Tue",2024-10-01 22:14,-283.5,₽,-283.5,₽,YM*weasel g,5037,Personal,Subscriptions,None,None,None
1788,057b5e4eb3,"07.11, Thu",2024-11-07 7:28,-294,₽,-294,₽,YM*weasel g,5037,Personal,Subscriptions,None,None,None
467,fe2159d20c,"14.08, Wed",2024-08-14 10:29,-283.5,₽,-283.5,₽,YM*weasel g,2475,Personal,Subscriptions,None,None,None


In [106]:
similar_indices

array([1068, 1018, 1318, ..., 1219, 1220,    0])

In [77]:
similarity[similarity.argsort()[::-1]]

array([1., 1., 1., ..., 0., 0., 0.])

In [78]:
similar_labeled

,id,Дата операции,Время операции,Сумма операции в валюте карты,Валюта карты,Сумма в валюте операции,Валюта операции,Описание операции,Номер карты,Категория,Подкатегория,Долг,Хэштэг,Комментарий
25,2aa0d2b5a4,"26.10, Sat",2024-10-26 9:33,-3450,₽,-3450,₽,OCHAKOVO,1665,Personal,Wellness,None,None,None


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Фильтруем транзакции без категории
missing_category = transactions_df[transactions_df['Категория'].isnull()]

# Транзакции с уже установленной категорией
labeled_data = transactions_df[transactions_df['Категория'].notnull()]

# Применим TF-IDF для текстового анализа
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(transactions_df['Описание операции'])

# Создадим копию данных для заполнения категорий
transactions_filled = transactions_df.copy()

# Функция для определения наиболее частой категории и подкатегории
def assign_category(index, tfidf_matrix, labeled_data):
    # Сравним текущую транзакцию с другими
    similarity = cosine_similarity(tfidf_matrix[index], tfidf_matrix).flatten()
    
    # Отсортируем по схожести
    similar_indices = similarity.argsort()[::-1][1:6]  # 5 наиболее похожих
    similar_transactions = transactions_df.iloc[similar_indices]
    
    # Фильтруем только те, у которых есть категория
    similar_labeled = similar_transactions[similar_transactions['Категория'].notnull()]
    
    if not similar_labeled.empty:
        # Находим наиболее частую категорию и подкатегорию
        category = similar_labeled['Категория'].mode().iloc[0]
        subcategory = similar_labeled['Подкатегория'].mode().iloc[0]
        return category, subcategory
    else:
        return None, None

# Проставим категории для транзакций с пропущенными значениями
for index in missing_category.index:
    category, subcategory = assign_category(index, tfidf_matrix, labeled_data)
    if category:
        transactions_filled.at[index, 'Категория'] = category
        transactions_filled.at[index, 'Подкатегория'] = subcategory

# Покажем результат
# import ace_tools as tools; tools.display_dataframe_to_user(name="Transactions with Filled Categories", dataframe=transactions_filled)


In [7]:
transactions_filled.head()

,id,Дата операции,Время операции,Сумма операции в валюте карты,Валюта карты,Сумма в валюте операции,Валюта операции,Описание операции,Номер карты,Категория,Подкатегория,Долг,Хэштэг,Комментарий
0,701a707742,"30.10, Wed",2024-10-30 20:27,-4000,₽,-4000,₽,Перевод на карту другого банка,5037,Transfer,Internal,None,None,None
1,675c171212,"30.10, Wed",2024-10-30 12:57,-2283.1,₽,-2283.1,₽,Оплата в Yandex.Lavka Moskva RUS,2475,Food & Dining,Delivery,None,None,None
2,2ef6d9a28b,"30.10, Wed",2024-10-30 11:39,-2730,₽,-2730,₽,Оплата в EGGSELLENT MOSKVA RUS,1665,Entertainment,Restaurants,None,None,None
3,dd282173e5,"29.10, Tue",2024-10-29 21:53,-900,₽,-900,₽,Внутренний перевод на договор,5427,Entertainment,Restaurants,None,None,None
4,dcb8f77afa,"29.10, Tue",2024-10-29 20:49,-50,₽,-50,₽,Внутренний перевод на договор,5044,Sport,Padel,None,None,None


In [8]:
transactions_merged = transactions_df.merge(transactions_filled[['id', 'Категория', 'Подкатегория']], on="id", how="left", suffixes=(None, "_filled"))

In [13]:
transactions_merged.loc[transactions_merged['Категория'].isnull()].sort_values(by="Время операции", ascending=False)[["Время операции", "Сумма операции в валюте карты", "Описание операции", 
                                                                       'Категория', 'Подкатегория',
                                                                       'Категория_filled', 'Подкатегория_filled']].head(50)

,Время операции,Сумма операции в валюте карты,Описание операции,Категория,Подкатегория,Категория_filled,Подкатегория_filled
1815,2024-11-13 17:29,-2630,Оплата в FABRIKA KUKHNYA MOSKVA RUS,None,None,Entertainment,Carsharing
1864,2024-11-04 19:01,-434,Оплата в НЕТМОНЕТ,None,None,Entertainment,Other
1870,2024-11-03 11:57,-2969.1,Оплата в Yandex.Lavka Moskva RUS,None,None,Food & Dining,Delivery
1872,2024-11-02 19:49,-2665,Оплата в LEONARDO Gorod Moskva RUS,None,None,Transportation,Carsharing
1874,2024-11-02 11:58,-10000,Перевод на карту другого банка,None,None,Transfer,Internal
1875,2024-11-02 11:09,-251.5,Оплата в мскАпт1741_P_QR,None,None,Transportation,Carsharing
1877,2024-11-01 10:48,-120000,Внешний перевод по номеру телефона +79104528167,None,None,Household,Mortgage/Rent
1879,2024-10-31 23:29,2300,Внутрибанковский перевод с договора,None,None,Transfer,Debt
1881,2024-10-31 21:41,124000,Внутрибанковский перевод с договора,None,None,Transfer,Debt
141,2024-10-08 18:00,-1700,Оплата в PE SPIRKOV IVAN SERGEEVICH g Moskva RU,None,None,Household,Groceries
